# Tulya Experiment 2 — Ruthless Cross-Domain Falsification

This notebook tests the **pre-registered claim**, not a moving hypothesis.

Before running, read:
- `EXPERIMENT2_PREREGISTRATION.md`
- `EXPERIMENT2_DATA_ADEQUACY.md`

The core kill gates are frozen in Git history. If they fail, the notebook's correct output is **KILL_PRODUCT_DIRECTION**.

**Kaggle settings:** enable a GPU and turn Internet ON (Fashion-MNIST and CIFAR-10 are downloaded by torchvision).

In [ ]:
import os, sys, subprocess, json, time
REPO = "/kaggle/working/tulya-training-dynamics"
if os.path.exists(REPO):
    subprocess.run(["git","-C",REPO,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","https://github.com/Vedsaga/tulya-training-dynamics.git",REPO], check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import torch, pandas as pd
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a Kaggle GPU before continuing.")

## 1. Confirm the frozen rules

This cell only prints the preregistration. Do not edit thresholds in the notebook.

In [ ]:
print(open("EXPERIMENT2_PREREGISTRATION.md").read())
print("\n--- DATA ADEQUACY ADDENDUM ---\n")
print(open("EXPERIMENT2_DATA_ADEQUACY.md").read())

## 2. Small engineering smoke test

This is **not** part of the 80-run dataset. It checks that training, telemetry, saving, and event labeling execute on Kaggle. If this crashes, fix the engineering bug before the full suite; do not alter the hypothesis or thresholds.

In [ ]:
from experiment2_core import RunSpec, run_one

SMOKE_ROOT="/kaggle/working/tulya_exp2_smoke"
smoke = run_one(
    RunSpec("synthetic_sequence_gru","healthy",98765),
    SMOKE_ROOT,
)
smoke

## 3. Run the full preregistered corpus

This is 4 domains × 5 inducing configurations × 4 seeds = **80 runs**.

The runner is resumable. If Kaggle disconnects, rerun this cell; completed run summaries are reused.

Do not run the blind evaluator until this finishes.

In [ ]:
from experiment2_core import run_suite

ROOT="/kaggle/working/tulya_exp2"
start=time.time()
manifest = run_suite(ROOT, seeds=(0,1,2,3), resume=True)
print("suite wall time (this invocation):", time.time()-start)
display(manifest.groupby(["domain","event"]).size().rename("runs").reset_index())
display(manifest)

## 4. Data-adequacy gate

This does **not** judge the hypothesis. It checks whether our inducing configurations actually produced enough heterogeneous outcomes to make the blind test meaningful.

If it prints `INVALID_DATA_GENERATION`, stop. Under the preregistration, exactly one **config-only** repair is allowed; the telemetry, evaluator and thresholds remain frozen.

In [ ]:
event_counts = manifest.groupby(["domain","event"]).size()
per_domain_classes = manifest.groupby("domain")["event"].nunique()
global_classes = manifest["event"].nunique()

adequate = bool((per_domain_classes >= 2).all() and global_classes >= 3)
print("event classes per domain:")
display(per_domain_classes)
print("global event classes:", global_classes)

if not adequate:
    print("\nVERDICT: INVALID_DATA_GENERATION")
    print("Do NOT run the blind evaluator. Send me manifest.csv and we apply the one preregistered config-only repair.")
else:
    print("\nDATA ADEQUACY: PASS — blind evaluation is allowed.")

## 5. Blind leave-one-domain-out evaluation

Run **only if data adequacy passed**.

A = learning curves  
B = strong raw telemetry  
C = normalized/canonical dynamics

The evaluator trains on three complete domains and tests zero-shot on the fourth. It uses only prefixes ending at 10%, 20%, 30%, or 40% of the run and excludes cases where the event already happened.

In [ ]:
if not adequate:
    raise RuntimeError("Data adequacy failed; evaluator intentionally blocked.")

from experiment2_eval import build_table, evaluate

forecast_table = build_table(ROOT)
print("forecast prefixes:", len(forecast_table))
folds, result = evaluate(ROOT)

display(folds)
print(json.dumps(result, indent=2))

## 6. Instrumentation-overhead gate

This is an isolated paired run and is not used by the predictor. Small models can exaggerate fixed monitoring overhead; the preregistered Experiment-2 engineering target is ≤2%. Core transfer/value gates remain the product kill gates.

In [ ]:
from experiment2_core import benchmark_overhead
overhead = benchmark_overhead(ROOT, domain="synthetic_sequence_gru", seed=991)
overhead

## 7. Final decision

No reinterpretation:

- If the core transfer/value gates fail: **KILL_PRODUCT_DIRECTION**
- If core gates pass but an engineering gate fails: one engineering correction is allowed without changing the feature hypothesis.
- If the core gates pass: **CONTINUE_TO_EXPERIMENT_3**, where the predictor is frozen and tested on a genuine small LM/SFT workload.

In [ ]:
print("BLIND VERDICT:", result["verdict"])
print("OVERHEAD PASS:", overhead["pass_le_0_02"])

if result["verdict"] == "KILL_PRODUCT_DIRECTION":
    print("\nSTOP. Do not invent another metric.")
elif not overhead["pass_le_0_02"]:
    print("\nCore hypothesis survived, but the monitoring implementation misses the preregistered overhead gate.")
else:
    print("\nExperiment 2 survived. Next and only next: freeze predictor -> real small-LM/SFT transfer test.")

print("\nArtifacts to send me:")
for name in ["manifest.csv","forecast_table.csv","evaluation_folds.csv","evaluation_summary.json"]:
    print(os.path.join(ROOT,name))